# Clifford Tableaux in Zixy

The `zixy` project aims to build a library for high-performance manipulation of Pauli strings, binary strings, and other operators for use cases in quantum chemistry, circuit simulation, compilation, and beyond, capable of running at scales of thousands of qubits. Python bindings are provided for convenient access. For the purposes of this demo, we will be focussing on it as a library for computing with Pauli strings.

## Pauli strings

A _Pauli string_ $P \in \left\{I, X, Y, Z\right\}^{\otimes n}$ is a tensor product of some number of _Pauli matrices_:

$$\begin{align*}
I &= \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} &
X &= \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \\
Y &= \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix} &
Z &= \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}
\end{align*}$$

The main data structures provided by `zixy` are collections of Pauli strings (tensor products of the Pauli matrices $I, X, Y, Z$) which can be taxonomised by:
- Scalars
  - No scalar
  - Signs $\{ +1, -1 \}$
  - Complex signs $\{ +1, +i, -1, -i\}$
  - Real numbers
  - Complex numbers
- Collection
  - Single string (a.k.a. term)
  - Ordered list
  - Set (ordered list with uniqueness enforced)
  - Sums (represents a weighted sum of the matrices)

|      | None     | Sign       | CSign             | Real       | Complex       |
|------|----------|------------|-------------------|------------|---------------|
| Term | `String` | `SignTerm` | `ComplexSignTerm` | `RealTerm` | `ComplexTerm` |
| List | `List`   | `SignList` | `ComplexSignList` | `RealList` | `ComplexList` |
| Set  | `Set`    | `SignSet`  | `ComplexSignSet`  | `RealSet`  | `ComplexSet`  |
| Sum  |          |            |                   | `RealSum`  | `ComplexSum`  |

For this tutorial, we will start by working with individual terms to get the basics before focussing on `SignList`.

In [1]:
from zixy.qubit import Qubits
from zixy.qubit.pauli import RealTermSum

# Specify a Pauli string by pairs of Pauli letter I/X/Y/Z and qubit index number 0/1/2/...
y = RealTermSum.from_str("Y0", Qubits.from_count(1))
print(y)
print(y.to_sparse_matrix(False).toarray())

(1.0, Y0)
[[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]


In [2]:
from zixy.qubit.pauli import ComplexTerm

# When giving a scalar, write the term as a pair
zz = ComplexTerm.from_str("(4.18-.84j, Z0 Z1)", Qubits.from_count(2))
print(zz)

((4.18-0.84j), Z0 Z1)


In [3]:
from zixy.qubit.pauli import SignTerm

# We can omit I's from the list so long as we specify the number of qubits expected
xi = SignTerm.from_str("(-1, X0)", Qubits.from_count(2))
print(xi)

(-1, X0)


In [4]:
from zixy.qubit.pauli import I, X, Y, Z

# Object can be more convenient than strings when constructing strings programmatically
ixyz = SignTerm(qubits=Qubits.from_count(4), source=[I, X, Y, Z])
print(ixyz)

(+1, X1 Y2 Z3)


## Multiplication

Using the interchange law
$$
\left( A \otimes B \right) \cdot \left( C \otimes D \right) = \left( A \cdot C \right) \otimes \left( B \cdot D \right)
$$
a product of two Pauli strings can be computed component-wise, accumulating the scalars as needed.

Remember that
$$\begin{align*}
I \cdot I &= I & I \cdot X &= X & I \cdot Y &= Y & I \cdot Z &= Z \\
X \cdot I &= X & X \cdot X &= I & X \cdot Y &= iZ & X \cdot Z &= -iY \\
Y \cdot I &= Y & Y \cdot X &= -iZ & Y \cdot Y &= I & Y \cdot Z &= iX \\
Z \cdot I &= Z & Z \cdot X &= iY & Z \cdot Y &= -iX & Z \cdot Z &= I
\end{align*}$$

In `zixy`, we can only multiply objects of the same type. A multiplication will also be rejected if the resulting scalar does not match the scalar type.

In [5]:
from zixy.qubit.pauli import SignTerm, ComplexSignTerm

## This will fail because the types don't match
# print(String.from_str("X0", Qubits(1)) * ComplexTerm.from_str("(1, Y0)", Qubits(1)))

## X.Y = iZ, so this will fail because we can't record the i
# print(String.from_str("X0", Qubits(1)) * String.from_str("Y0", Qubits(1)))

## Multiplying two commuting Pauli strings always generates +-1 as the scalar, so SignTerm is sufficient
## X.Z = -iY, Y.Z = iX, (+1) * (-1) * (-i) * (+i) = -1
print(
    SignTerm.from_str("(+1, X0 Y1)", Qubits.from_count(2))
    * SignTerm.from_str("(-1, Z0 Z1)", Qubits.from_count(2))
)

## ComplexSignTerm will always be able to handle the scalars
## X.Z = -iY, X.I = X, (+1) * (+i) * (-i) * (+1) * (-i) = -i
print(
    ComplexSignTerm.from_str("(+1, X0 X1 X2)", Qubits.from_count(3))
    * ComplexSignTerm.from_str("(+i, Z0 Z2)", Qubits.from_count(3))
)

(-1, Y0 X1)
(-i, Y0 X1 Y2)


### Exercise

1. The above multiplication table enumerates all options for multiplying 1-qubit Pauli strings. Write a loop that constructs every 2-qubit Pauli string using `ComplexSignTerm.from_list()`. Use this to print out the entire multiplication table for every pair of 2-qubit Pauli strings (it should print out $4 \times 4 \times 4 \times 4 = 256$ entries).

In [6]:
def all2qubitpaulis():
    for p0 in [I, X, Y, Z]:
        for p1 in [I, X, Y, Z]:
            yield ComplexSignTerm(source=[p0, p1], qubits=Qubits.from_count(2))


ps0 = list(all2qubitpaulis())
ps1 = list(all2qubitpaulis())

for ps_a in ps0:
    for ps_b in ps1:
        print(f"{ps_a} * {ps_b} = {ps_a * ps_b}")

(+1, ) * (+1, ) = (+1, )
(+1, ) * (+1, X1) = (+1, X1)
(+1, ) * (+1, Y1) = (+1, Y1)
(+1, ) * (+1, Z1) = (+1, Z1)
(+1, ) * (+1, X0) = (+1, X0)
(+1, ) * (+1, X0 X1) = (+1, X0 X1)
(+1, ) * (+1, X0 Y1) = (+1, X0 Y1)
(+1, ) * (+1, X0 Z1) = (+1, X0 Z1)
(+1, ) * (+1, Y0) = (+1, Y0)
(+1, ) * (+1, Y0 X1) = (+1, Y0 X1)
(+1, ) * (+1, Y0 Y1) = (+1, Y0 Y1)
(+1, ) * (+1, Y0 Z1) = (+1, Y0 Z1)
(+1, ) * (+1, Z0) = (+1, Z0)
(+1, ) * (+1, Z0 X1) = (+1, Z0 X1)
(+1, ) * (+1, Z0 Y1) = (+1, Z0 Y1)
(+1, ) * (+1, Z0 Z1) = (+1, Z0 Z1)
(+1, X1) * (+1, ) = (+1, X1)
(+1, X1) * (+1, X1) = (+1, )
(+1, X1) * (+1, Y1) = (+i, Z1)
(+1, X1) * (+1, Z1) = (-i, Y1)
(+1, X1) * (+1, X0) = (+1, X0 X1)
(+1, X1) * (+1, X0 X1) = (+1, X0)
(+1, X1) * (+1, X0 Y1) = (+i, X0 Z1)
(+1, X1) * (+1, X0 Z1) = (-i, X0 Y1)
(+1, X1) * (+1, Y0) = (+1, Y0 X1)
(+1, X1) * (+1, Y0 X1) = (+1, Y0)
(+1, X1) * (+1, Y0 Y1) = (+i, Y0 Z1)
(+1, X1) * (+1, Y0 Z1) = (-i, Y0 Y1)
(+1, X1) * (+1, Z0) = (+1, Z0 X1)
(+1, X1) * (+1, Z0 X1) = (+1, Z0)
(+1, X1) * (+1

## Stabilizer Tableau

Here are some facts about stabilizers we have already established:
- If $P$ and $Q$ are both stabilizers of a state, then they _commute_ $P \cdot Q = Q \cdot P$.
- The stabilizers of a state form an _abelian group_, and so can be described by a set of _generators_.
- Given $n$ independent, commuting Pauli strings on $n$ qubits along with any choices of phases for them, there is a unique state stabilized by all of them.

A _stabilizer tableau_ is a list of Pauli strings and associated phases, describing one possible set of generators for a particular quantum state. This allows us to use it to represent a quantum state. We will be using the `SignList` class to capture this.

### Exercise

2. The full statevector for an $n$-qubit state would require $2^n$ complex numbers to store. How many bits of information are required to store a stabilizer tableau for an $n$-qubit state?

In [7]:
from zixy.qubit.pauli import SignTerms

tab = SignTerms(Qubits.from_count(3))
tab.append([X, Y, Z])
tab.append([Z, X, Z])
tab.append([X, X, X])
print(tab)

(+1, X0 Y1 Z2), (+1, Z0 X1 Z2), (+1, X0 X1 X2)


How can we figure out what state this corresponds to? We know that if we combine the projectors for these stabilizers then we get a projector onto the state.

Remember that the projector onto $(-1)^a P$ is given by $\frac{1}{2}\left( I + \left( -1 \right)^a P \right)$.

In [8]:
# from numpy import eye
#
## Start with the identity
# combined_projector = eye(8, dtype=complex)
# for i in range(len(tab)):
#    # For each stabilizer in the tableau, construct the projector
#    proj = 0.5 * (eye(8, dtype=complex) + tab[i].to_sparse_matrix(False))
#    # Multiply it onto our cumulative projector
#    combined_projector = combined_projector * proj
# print(combined_projector)

### Exercise

3. We know what the eigenvectors for this projector should be: +1 for the state we are representing, and 0 for all states orthogonal to it. Use the tools in numpy to produce the spectral decomposition of this projector or solve the eigenvector equation directly for the +1 eigenvector. Repeat the process to find the 4-qubit state corresponding to $(-1, Z_0 Z_1 Z_3), (+1, Z_0 Z_1 Z_2), (+1, Z_0 Z_2 Z_3), (-1, Z_0 Z_1 Z_2 Z_3)$.

In [9]:
# from numpy import linalg
#
# eigenvalues, eigenvectors = linalg.eig(combined_projector)
# print(eigenvalues.round(3))
# print(eigenvectors[:, 1].round(3))

In [10]:
# tab = SignTerms(Qubits.from_count(4))
#
#
# tab.append_term(SignTerm.from_str("(-1, Z0 Z1 Z3)"))
# tab.append_term(SignTerm.from_str("(1, Z0 Z1 Z2)"))
# tab.append_term(SignTerm.from_str("(1, Z0 Z2 Z3)"))
# tab.append_term(SignTerm.from_str("(-1, Z0 Z1 Z2 Z3)"))
#
# combined_projector = eye(16, dtype=complex)
# for i in range(len(tab)):
#    proj = 0.5 * (eye(16, dtype=complex) + tab[i].to_sparse_matrix(False))
#    combined_projector = combined_projector * proj
#
# eigenvalues, eigenvectors = linalg.eig(combined_projector)
# print(eigenvalues.round(3))
# print(eigenvectors[:, 11].round(3))

Because the stabilizers form a group, the product of any two stabilizers is itself a stabilizer.

`tab[i] *= tab[j]` updates the $i$ th stabilizer in the tableau with the product of itself with the $j$ th stabilizer. Whenever `i != j`, this preserves the independence of the set of stabilizers in the tableau. This new tableau gives us a different choice of generators for the same group as before. Because the stabilizer group has not changed, the state it corresponds to should be the same!

### Exercise

4. Do some multiplications between the terms in the tableau from exercise 3 and find the combined projector and state again to verify that they are unchanged.

In [11]:
# tab[1] *= tab[0]
# tab[2] *= tab[0]
# tab[3] *= tab[0]
# tab[1] *= tab[2]
# tab[0] *= tab[1]
# tab[2] *= tab[1]
# tab[3] *= tab[2]
# tab[1] *= tab[3]
# tab[2] *= tab[3]
# print(tab)

In [12]:
# combined_projector = eye(16, dtype=complex)
# for i in range(len(tab)):
#    proj = 0.5 * (eye(16, dtype=complex) + tab[i].to_sparse_matrix(False))
#    combined_projector = combined_projector * proj
#
# eigenvalues, eigenvectors = linalg.eig(combined_projector)
# print(eigenvalues.round(3))
# print(eigenvectors[:, 11].round(3))

## Canonicalisation of a Tableau

We have established that the state is unchanged when we update the stabilizers in the tableau by multiplication with one another. Linear equation systems also have this kind of property, where the solutions to the equations are preserved when adding equations to one another. If we write the linear equation system in its matrix form $Ax = B$, this corresponds to taking row combinations in the matrices $A$ and $B$. Such linear equation systems can be put into a canonical form by _Gaussian elimination_, so can we do the same with stabilizer tableau? Yes!

The loop for Gaussian elimination of a real matrix iterates over the columns:
- Find a _pivot_ - a row with a non-zero entry in that column.
- Normalise that row to make the pivot entry 1.
- For any other rows with a non-zero entry in that column, subtract a scaled multiple of the pivot row to cancel the entry.

When working with a stabilizer tableau, we iterate over the _qubits_:
- Find a pivot - a stabilizer with an $X$ or $Y$ entry on that qubit.
- Normalisation is not required.
- For any other stabilizer with $X$ or $Y$ on that qubit, multiply with the pivot to reduce it to $I$ or $Z (= iY \cdot X)$.
- Now there is at most one stabilizer with $X$ or $Y$ on that qubit, but there may still exist some with $Z$, so repeat the process for them.

Let's walk through those steps with an example.

In [13]:
from zixy.qubit.pauli import X, Y, Z, I
from zixy.qubit.pauli import SignTerms
from zixy.qubit import Qubits

tab = SignTerms(Qubits.from_count(6))
tab.append([X, Z, I, I, I, I])
tab.append([Y, X, Z, Z, I, X])
tab.append([I, Z, Y, I, X, X])
tab.append([X, I, Y, Y, Z, Y])
tab.append([Z, X, X, I, I, I])
tab.append([I, I, X, Y, Z, X])
# Use `.to_dataframe()` to get a summary of the entire tableau
tab.to_dataframe()

,Component,Coefficient
0,X0 Z1,+1
1,Y0 X1 Z2 Z3 X5,+1
2,Z1 Y2 X4 X5,+1
3,X0 Y2 Y3 Z4 Y5,+1
4,Z0 X1 X2,+1
5,X2 Y3 Z4 X5,+1


In [14]:
from copy import copy


tab_copy = copy(tab)

For qubit 0, we have $X_0$ or $Y_0$ entries in stabilizers 0, 1, and 3. Picking 0 as our pivot, we can then reduce stabilizer 1 to a $Z_0$ entry and eliminate the $X$ in stabilizer 3.

In [15]:
tab[1] *= tab[0]
tab[3] *= tab[0]
tab.to_dataframe()

,Component,Coefficient
0,X0 Z1,+1
1,Z0 Y1 Z2 Z3 X5,-1
2,Z1 Y2 X4 X5,+1
3,Z1 Y2 Y3 Z4 Y5,+1
4,Z0 X1 X2,+1
5,X2 Y3 Z4 X5,+1


Now we have $Z_0$ entries in stabilizers 1 and 4. We can pick stabilizer 1 as our new pivot and eliminate the $Z_0$ in stabilizer 4.

In [16]:
tab[4] *= tab[1]
tab.to_dataframe()

,Component,Coefficient
0,X0 Z1,+1
1,Z0 Y1 Z2 Z3 X5,-1
2,Z1 Y2 X4 X5,+1
3,Z1 Y2 Y3 Z4 Y5,+1
4,Z1 Y2 Z3 X5,-1
5,X2 Y3 Z4 X5,+1


We now have exactly one stabilizer with $X_0$ and one with $Z_0$ so we move onto qubit 1. Of the remaining stabilizers, none contain $X_1$ or $Y_1$ so we pivot on stabilizer 2 to eliminate $Z_1$'s. This includes eliminating the $Z_1$ in stabilizer 0. Since stabilizer 1 contains $Y_1$, we will also prefer it to be either $X_1$ or $Z_1$ if possible, so we will also multiply that by our pivot.

In [17]:
tab[0] *= tab[2]
tab[1] *= tab[2]
tab[3] *= tab[2]
tab[4] *= tab[2]
tab.to_dataframe()

,Component,Coefficient
0,X0 Y2 X4 X5,+1
1,Z0 X1 X2 Z3 X4,-1
2,Z1 Y2 X4 X5,+1
3,Y3 Y4 Z5,+1
4,Z3 X4,-1
5,X2 Y3 Z4 X5,+1


For qubit 2, the only choice for a pivot is stabilizer 5 which contains $X_2$. To keep things looking vaguely upper diagonal, we will first move it to index 3.

In [18]:
# Swaps the stabilizers 3 and 5, see exercise 5
tab[3] *= tab[5]
tab[5] *= tab[3]
tab[3] *= tab[5]
tab.to_dataframe()

,Component,Coefficient
0,X0 Y2 X4 X5,+1
1,Z0 X1 X2 Z3 X4,-1
2,Z1 Y2 X4 X5,+1
3,X2 Y3 Z4 X5,+1
4,Z3 X4,-1
5,Y3 Y4 Z5,+1


And now we can eliminate the $X_2$ in stabilizer 1 and reduce $Y_2$'s to $Z_2$'s in stabilizers 0 and 2.

In [19]:
tab[0] *= tab[3]
tab[1] *= tab[3]
tab[2] *= tab[3]
tab.to_dataframe()

,Component,Coefficient
0,X0 Z2 Y3 Y4,-1
1,Z0 X1 X3 Y4 X5,+1
2,Z1 Z2 Y3 Y4,-1
3,X2 Y3 Z4 X5,+1
4,Z3 X4,-1
5,Y3 Y4 Z5,+1


We continue this practice with qubit 3 until every stabilizer has been used as a pivot.

In [20]:
# 5 contains Y3, move it to 4
tab[4] *= tab[5]
tab[5] *= tab[4]
tab[4] *= tab[5]
# X3s and Y3s in 0, 1, 2, and 3
tab[0] *= tab[4]
tab[1] *= tab[4]
tab[2] *= tab[4]
tab[3] *= tab[4]
tab.to_dataframe()

,Component,Coefficient
0,X0 Z2 Z5,-1
1,Z0 X1 Z3 Y5,+1
2,Z1 Z2 Z5,-1
3,X2 X4 Y5,-1
4,Y3 Y4 Z5,+1
5,Z3 X4,-1


In [21]:
# 5 contains Z3, as does 1, and reduce the Y3 in 4 to X3
tab[1] *= tab[5]
tab[4] *= tab[5]
# This is our final tableau
tab.to_dataframe()

,Component,Coefficient
0,X0 Z2 Z5,-1
1,Z0 X1 X4 Y5,-1
2,Z1 Z2 Z5,-1
3,X2 X4 Y5,-1
4,X3 Z4 Z5,-1
5,Z3 X4,-1


In [22]:
print(type(tab[5]))

<class 'zixy.qubit.pauli._terms.SignTerm'>


We claim this is our canonical representation of the tableau with respect to the given ordering of the qubits (changing the qubit order changes which stabilizers we pivot on and therefore the final tableau). We know each multiplication update preserves the quantum state we are representing, this gives us a canonical, efficient representation of the state. The uniqueness of this form means we can use it as a way to test whether two tableaus describe the same state.

### Exercise

5. In the above code example, we swapped two Pauli strings using a sequence of multiplications. Prove that these multiplications will successfully swap an arbitrary pair of commuting Pauli strings.

6. Write a function that takes a `SignList` representing a stabilizer tableau and applies this Gaussian elimination routine. HINT: To check a particular entry of the tableau, use `tab[stab_index].get_pauli(qubit_index)` and compare it to the builtin `I`, `X`, `Y`, and `Z`.

In [23]:
# P
# Q
# P_1 := P * Q
# Q_1 := Q * P_1
# P_2 := P_1 * Q_1

In [32]:
# TODO: use SignTerms.canonicalize_all() instead here

def canonicalise(tab: SignTerms) -> None:
    pivot_row = 0
    for pivot_col in range(len(tab.qubits)):
        # Solve Xs
        for i in range(pivot_row, len(tab)):
            p = tab[i].string[pivot_col]
            # Find a stabilizer to pivot on
            if p == X or p == Y:
                # Guarantee that pivot_row contains X or Y
                if i != pivot_row:
                    tab[pivot_row] *= tab[i]
                # Eliminate X or Y from all other stabilizers
                for j in range(len(tab)):
                    pj = tab[j].string[pivot_col]
                    if j != pivot_row and (pj == X or pj == Y):
                        tab[j] *= tab[pivot_row]
                pivot_row += 1
                break
        # Do the same for Zs
        for i in range(pivot_row, len(tab)):
            p = tab[i].string[pivot_col]
            if p == Z:
                if i != pivot_row:
                    tab[pivot_row] *= tab[i]
                for j in range(len(tab)):
                    pj = tab[j].string[pivot_col]
                    if j != pivot_row and (pj == Z or pj == Y):
                        tab[j] *= tab[pivot_row]
                pivot_row += 1
                break

In [25]:
print(tab_copy.to_dataframe())
canonicalise(tab_copy)
print(tab_copy.to_dataframe())

     Component Coefficient
0     X0 Z2 Z5          -1
1  Z0 X1 X4 Y5          -1
2     Z1 Z2 Z5          -1
3     X2 X4 Y5          -1
4     X3 Z4 Z5          -1
5        Z3 X4          -1
     Component Coefficient
0     X0 Z2 Z5          -1
1  Z0 X1 X4 Y5          -1
2     Z1 Z2 Z5          -1
3     X2 X4 Y5          -1
4     X3 Z4 Z5          -1
5        Z3 X4          -1


In [26]:
# def gauss(mat: np.ndarray):
#     row_to_solve = 0
#     for i in range(0, mat.cols()):
#         found = False
#         for j in range(row_to_solve, mat.rows()):
#             if mat(i, j) != 0:
#                 mat.row(j).swap(mat.row(row_to_solve))
#                 mat.row(row_to_solve) *= 1/(mat(i, row_to_solve))
#                 found = True
#                 break
#         if found:
#             for j in range(row_to_solve + 1, mat.rows()):
#                 mat.row(j) -= mat.row(row_to_solve) * mat(i, j)
#             row_to_solve += 1


In [27]:
# def canonicalise(tab: SignList):
#     row_to_solve = 0
#     for i in range(0, tab.num_qubits):
#         # solve the X
#         found_x = False
#         for j in range(row_to_solve, tab.num_rows):
#             if tab[i, j] in (X,Y):
#                 tab.row(j).swap(tab.row(row_to_solve))
#                 found_x = True
#                 break
#         if found_x:
#             for j in range(0, tab.num_rows):
#                 if j == row_to_solve:
#                     continue
#                 if tab[i, j] in (X,Y):
#                     tab.row(j) *= tab.row(row_to_solve)
#             row_to_solve += 1

#         # solve the Z
#         found_z = False
#         for j in range(row_to_solve, tab.num_rows):
#             if tab[i, j] == Z:  # "X-iness" already eliminated above
#                 tab.row(j).swap(tab.row(row_to_solve))
#                 found_z = True
#                 break
#         if found_z:
#             for j in range(0, tab.num_rows):
#                 if j == row_to_solve:
#                     continue
#                 if tab[i, j] == Z:
#                     tab.row(j) *= tab.row(row_to_solve)
#             row_to_solve += 1


## The Symplectic Representation of Paulis

Gaussian elimination guarantees us an upper-diagonal matrix with precisely one row per leading column. The canonical tableau looks similar to that form, but we now have up to two stabilizers per leading qubit.

|   | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| - | X |   | Z |   |   | Z |
| - | Z | X |   |   | X | Y |
| + |   | Z | Z | Z | X | Z |
| - |   |   | X |   | X | Y |
| + |   |   |   | X | Z | Z |
| - |   |   |   | Z | X |   |

There is a way to view this tableau as a proper upper-diagonal matrix with one row per leading column. Since there are 4 Pauli matrices, we can use 2 bits to represent each one with the following mapping:
$$\begin{align*}
(0, 0) &\leftrightarrow I & (0, 1) &\leftrightarrow Z \\
(1, 0) &\leftrightarrow X & (1, 1) &\leftrightarrow Y
\end{align*}$$
Since $Y = i X \cdot Z$, this mapping is nice since $(a, b)$ corresponds to $X^a \cdot Z^b$ up to some phase. Multiplication of Paulis (ignoring phase) also maps nicely into bit-wise XOR.
$$\left( X^{a_0} \cdot Z^{b_0} \right) \cdot \left( X^{a_1} \cdot Z^{b_1} \right) = \pm X^{a_0 + a_1} \cdot Z^{b_0 + b_1}$$
This mapping is referred to as the group homomorphism into _the Symplectic Group_. It's what enables us to store Pauli strings with very little memory and update them quickly. Picturing the above tableau in this representation (i.e. splitting each qubit column into separate $X$ and $Z$ columns and ignoring the phase column) shows that we have really done Gaussian elimination!

| X0 | Z0 | X1 | Z1 | X2 | Z2 | X3 | Z3 | X4 | Z4 | X5 | Z5 |
|----|----|----|----|----|----|----|----|----|----|----|----|
| 1  |    |    |    |    | 1  |    |    |    |    |    | 1  |
|    | 1  | 1  |    |    |    |    |    | 1  |    | 1  | 1  |
|    |    |    | 1  |    | 1  |    | 1  | 1  |    |    | 1  |
|    |    |    |    | 1  |    |    |    | 1  |    | 1  | 1  |
|    |    |    |    |    |    | 1  |    |    | 1  |    | 1  |
|    |    |    |    |    |    |    | 1  | 1  |    |    |    |

Many other libraries and papers will display tableaus using a binary encoding like this, sometimes with the $X$ and $Z$ columns for the same qubit next to each other, and sometimes grouping all of the $X$ columns first before all of the $Z$ columns. They may also transpose the tableau to display stabilizers in columns using rows for qubits; always check the surrounding content to see how to interpret the tableau.

### Exercise

7. Write a function that takes a `SignList` and displays it as a binary matrix (either by printing the bits directly or by converting to a `numpy` array of booleans).

In [28]:
def print_binary(tab: SignTerms) -> None:
    header = ""
    for _ in range(len(tab.qubits)):
        header += "XZ"
    header += " Ph"
    print(header)
    for i in range(len(tab)):
        line = ""
        for q in range(len(tab.qubits)):
            p = tab[i].string[q]
            if p == I:
                line += "00"
            elif p == X:
                line += "10"
            elif p == Y:
                line += "11"
            else:  # p == Z
                line += "01"
        if tab[i].coeff == 1:
            line += " 0"
        else:
            line += " 1"
        print(line)

In [29]:
print_binary(tab)

XZXZXZXZXZXZ Ph
100001000001 1
011000001011 1
000101000001 1
000010001011 1
000000100101 1
000000011000 1


## Clifford Gates

We can view any unitary $U$ as a change of basis from the computational basis into some other basis $\mathcal{B}$ - given a matrix $A$ describing how an operation acts wrt. the computational basis, $UAU^\dagger$ describes how it acts wrt. $\mathcal{B}$. We refer to $UAU^\dagger$ as the _conjugation of $A$ by_ $U$ (not to be confused with conjugation of complex numbers $(-)^*$).

Conjugation allows us to "push one matrix through another". If $U V$ describes performing $V$ then performing $U$, we can only reorder them to $V U$ if they commute, but we might be able to find a different $V'$ such that $V' U = U V$. Right-multiplying by $U^\dagger$ solves this as $V' = U V U^\dagger$.

We say that a unitary $U$ is _Clifford_ if, for every Pauli string $P$, $UPU^\dagger$ is a Pauli string (up to some phase).

### Exercises

Complete the following exercises on pen and paper before reading ahead to see how to use `zixy` to compute them.

8. Verify that the Hadamard gate is Clifford, i.e. show that $HIH$, $HXH$, $HYH$, and $HZH$ are all Paulis up to some phase.

9. Verify that $S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}$ is Clifford.

10. Prove that the $T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i \tfrac{\pi}{4}} \end{pmatrix}$ gate is not Clifford by finding a Pauli matrix $P \in \left\{I, X, Y, Z\right\}$ such that $TPT^\dagger$ is not a Pauli matrix up to any phase.

11. Conjugate each of $\left\{ X_c I_t, Z_c I_t, I_c X_t, I_c Z_t \right\}$ by the $\mathrm{C}X$ gate, where $c$ and $t$ refer to the indices of the control and target qubits of the gate. Why is this enough for us to claim that the $\mathrm{C}X$ gate is Clifford?

It turns out that any Clifford unitary can be synthesised as a circuit with only $H$, $S$, and $\mathrm{C}X$ gates. We will return to this in a future exercise sheet to show how to prove this, but for now it is just helpful to know that if we can conjugate Pauli strings by these three gates then we can compute the conjugations of any Clifford circuit. Given any sequence of $H$, $S$, and $\mathrm{C}X$ gates, `zixy` can conjugate Pauli strings very efficiently. A `GateList` specifies the order of the gates and which qubits they are applied to.

Y = |i><i| - |-i><-i|
HYH = H|i><i|H - H|-i><-i|H
= |-i><-i| - |i><i|
= -Y

H |i> = e^{i\pi/4} |-i>
/ 1 1  \ / 1 \ = / 1+i \ ~= / 1. \
\ 1 -1 / \ i /   \ 1-i /.   \ -i /

/ 1 1  \ / 0 -i \ / 1 1. \ = / i -i. \ / 1 1  \ = / 0  i \ = -Y
\ 1 -1 / \ i 0  / \ 1 -1 /.  \ -i -i / \ 1 -1 /.  \ -i 0 /

XZ = +-iY
HZH = X
HXH = Z
HYH ~= HXZH = HXHHZH = ZX ~= Y

UPU^dag
UQU^dag
UPQU^dag = UPU^dag UQU^dag

In [30]:
# from zixy.qubit.clifford import GateList, S, H, CX
#
# tab = SignTerms(Qubits.from_count(3))
# tab.append([Z, I, I])
# tab.append([I, Z, I])
# tab.append([I, I, Z])
# gate_seq = GateList.from_iterable([(H, 2), (CX, 2, 0), (CX, 2, 1), (S, 2)])
# tab.conj_clifford_list(gate_seq)
# tab.to_dataframe()

Given a Clifford unitary $U$, a stabilizer state $\left| \psi \right\rangle$, and a stabilizer tableau for $\left| \psi \right\rangle$, we can transform this into a tableau for $U \left| \psi \right\rangle$ by conjugating every stabilizer by $U$. For example, the above code starts with a tableau for $\left| 000 \right\rangle$, and computes the tableau for $S_2 \mathrm{C}X_{2, 1} \mathrm{C}X_{2, 0} H_2 \left| 000 \right\rangle$.

There are multiple ways to prove this; the following exercises will walk you through two different proof strategies. If you get stuck on one, try the other.

### Exercises

12. a. Show that, if $P$ stabilizes $\left| \psi \right\rangle$, then $UPU^\dagger$ stabilizes $U \left| \psi \right\rangle$.

    b. Using (a) or otherwise show that, if both $P$ and $Q$ are stabilizers of $\left| \psi \right\rangle$, then $UPU^\dagger$ and $UQU^\dagger$ commute.

    c. Show that $UPU^\dagger = I$ if and only if $P = I$.

    d. $\mathcal{P}$ is a complete set of stabilizers for $\left| \psi \right\rangle$ if it contains $n$ stabilizers (where $n$ is the number of qubits in our state) and no element can be written as a product of the others. Let $\mathcal{P'} = \left\{ UPU^\dagger \middle| P \in \mathcal{P} \right\}$. We know that every element of $\mathcal{P'}$ is a stabilizer of $U \left| \psi \right\rangle$. Prove, by contradiction, that no element of $\mathcal{P'}$ can be written as a product of the others.

13. a. The measurement projectors for a measurement of the Pauli string $P$ were $\frac{I \pm P}{2}$. Show that the measurement projectors for $UPU^\dagger$ are $U\frac{I \pm P}{2}U^\dagger$.

    b. Given a complete set of stabilizers $\mathcal{P}$ for $\left| \psi \right\rangle$, we know that the product of all measurement projectors from the strings in $\mathcal{P}$ gives $\left| \psi \right\rangle \left\langle \psi \right|$, the projector onto our state. Let $\mathcal{P'}$ be defined as in Exercise 12(d). Show that the product of all measurement projectors from $\mathcal{P'}$ gives the projector onto $U \left| \psi \right\rangle$.

`SignList.conj_clifford_list()` is implemented in the core rust library of `zixy` where it can benefit from the underlying Symplectic binary representation, but it works by defining functions to apply each kind of Clifford gate onto a tableau like the function below.

In [31]:
def apply_s(tab: SignTerms, qubit: int):
    for stab in range(tab.len()):
        pauli = tab[stab].get_pauli(qubit)
        if pauli == X:
            tab[stab].set_pauli(qubit, Y)
        elif pauli == Y:
            tab[stab].set_pauli(qubit, X)
            # Flip the phase on the stabilizer
            tab[stab].coeff *= -1
        # No need to do anything for I or Z

### Exercise

14. Write a pair of similar functions to update a tableau according to the application of an $H$ gate and a $\mathrm{C}X$ gate respectively. Test your implementation by simulating the generation of a Bell state as $\mathrm{C}X_{0,1} H_0 \left| 00 \right\rangle$.

Congratulations! You have just built your first Clifford simulator!

We can build on this knowledge in future exercises or in your own reading with the following topics:
- Simulating measurements on a stabilizer tableau.
- Circuit implementations of Pauli measurements.
- Mutual diagonalisation (performing commuting measurements at the same time).
- Stabilizer tableaus for unitaries (as opposed to tableaus for states presented here).
- Synthesising Clifford circuits from their tableaus (i.e. $H$, $S$, and $\mathrm{C}X$ are sufficient).
- Graph states and MBQC.
- Stabilizer codes for error detection and correction.